## In sample Study

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from strategies.supertrend_ema_confirmation.strategy import (
    SupertrendEmaConfirmationStrategy as Strategy,
)

## Constants

In [ ]:
from pathlib import Path

data_storage_path = Path.cwd().parent / "data"
backtest_results_dir = Path.cwd().parent / "backtest_results"
top_selection_path = backtest_results_dir / "top_selection"
backtest_results_dir_in_sample = backtest_results_dir / "in_sample"
backtest_results_dir_out_sample = backtest_results_dir / "out_sample"
reports_dir = Path.cwd().parent / "reports"
figures_dir = reports_dir / "figures"
time_frames = ["2h", "4h", "1d"]


## Study definition

A surviving param set should be robust along *both* axes through which a backtest can leak:

| Type | What changes | What stays the same | What it tests |
|------|-------------|---------------------|---------------|
| **A — Time OOS** | Date window (pre-2022: 2019–2021) | Universe (BTC/ETH on BITVAVO/EUR) | Do the params generalise to a different market regime? |
| **B — Universe OOS** | Symbols (LINK/AVAX/ATOM/ALGO/XRP) | Date window (2022–2025, same as in-sample) | Does the edge survive on assets the sweep never saw? |

Each regime runs as its own `Study` so the bundle envelope's per-study metadata and per-universe summaries are populated correctly — notebook 05 uses these to compare in-sample vs. OOS side by side.

---

### Type A — Time OOS

Tests generalisation to a different **market regime**: the 2019–2021 cycle includes the March 2020 crash and the late-2021 peak, neither of which the in-sample sweep saw. The window is strictly *before* the in-sample range so there is zero temporal overlap and no risk of leakage from the param sweep.

Universe is restricted to BTC and ETH because ADA, SOL and DOT had little or no BITVAVO history before 2021.

| Setting | Value |
|---------|-------|
| Windows | Rolling — train 365 d, test 180 d, gap 30 d, step 90 d |
| Symbols | BTC, ETH |
| Market / quote | BITVAVO / EUR |
| Engine | Vector |
| Risk-free rate | 2.7 % |

---

### Type B — Universe OOS

Tests generalisation to **unseen assets** while keeping the time period identical to the in-sample sweep. This isolates whether the strategy's edge is intrinsic to the signal or merely overfit to the original basket.

| Setting | Value |
|---------|-------|
| Windows | Rolling — train 365 d, test 180 d, gap 30 d, step 90 d |
| Symbols | LINK, AVAX, ATOM, ALGO, XRP |
| Market / quote | BITVAVO / EUR |
| Engine | Vector |
| Risk-free rate | 2.7 % |

In [ ]:
from datetime import datetime, timezone
from investing_algorithm_framework import generate_rolling_backtest_windows, Study, \
    BacktestEngine, Universe, show_study, StudySampleType

# Type A — Time OOS: 2019–2021, strictly before the in-sample range (2022–2025).
# Tests generalisation to a different market regime (March 2020 crash, 2021 peak).
time_oos_rolling_backtest_windows = generate_rolling_backtest_windows(
    start_date=datetime(2019, 1, 1, tzinfo=timezone.utc),
    end_date=datetime(2021, 12, 31, tzinfo=timezone.utc),
    train_days=365,
    test_days=180,
    gap_days=30,
    step_days=90,
)

time_oos_study = Study(
    name="time_oos_param_sweep",
    description="Time out-of-sample parameter sweep over the time_oos_basket universe "
    "(BTC/ETH on BITVAVO/EUR, 2019-01 → 2021-12).",
    risk_free_rate=0.027,
    initial_capital=1000,
    sample_type=StudySampleType.OUT_SAMPLE_TIME,
    universe=Universe(
        key="time_oos_basket",
        symbols=["BTC", "ETH"],
        trading_symbol="EUR",
        market='BITVAVO',
    ),
    backtest_windows=time_oos_rolling_backtest_windows,
    engines=[BacktestEngine.VECTOR],
)

show_study(time_oos_study)

# Type B — Universe OOS: same date window as in-sample (2022–2025), different symbols.
# Tests whether the edge survives on assets the sweep never saw.
universe_oos_rolling_backtest_windows = generate_rolling_backtest_windows(
    start_date=datetime(2022, 1, 1, tzinfo=timezone.utc),
    end_date=datetime(2025, 12, 30, tzinfo=timezone.utc),
    train_days=365,
    test_days=180,
    gap_days=30,
    step_days=90,
)

universe_oos_study = Study(
    name="universe_oos_param_sweep",
    description="Universe out-of-sample parameter sweep over the out_sample_basket universe "
    "(LINK/AVAX/ATOM/ALGO/XRP on BITVAVO/EUR, 2022-01 → 2025-12).",
    risk_free_rate=0.027,
    initial_capital=1000,
    sample_type=StudySampleType.OUT_SAMPLE_UNIVERSE,
    universe=Universe(
        key="out_sample_basket",
        symbols=["LINK", "AVAX", "ATOM", "ALGO", "XRP"],
        trading_symbol="EUR",
        market='BITVAVO',
    ),
    backtest_windows=universe_oos_rolling_backtest_windows,
    engines=[BacktestEngine.VECTOR],
)

show_study(universe_oos_study)


## Load Top Selection Params

The in-sample sweep persisted the surviving bundles to
`backtest_results/top_selection` via `prune_backtests(..., flatten=True)`.
Each bundle records the strategy params it was instantiated with under
`Backtest.metadata["params"]` (the `params` payload we attached when
calling `initialize_strategies` in notebook `03_param_sweep.ipynb`).

We rebuild the Tier-1 SQLite index over that folder, re-rank with the
same BALANCED focus, then materialise the winners through
`LocalDirStore` and pull each saved param dict back out via
`Backtest.get_metadata()` — no need to re-derive the grid combinations
by hand.

In [ ]:
from investing_algorithm_framework import BacktestEvaluationFocus, build_index, \
    rank_index, LocalDirStore, Study, StudySampleType

in_sample_study = Study(
    name="in_sample_param_sweep",
    sample_type=StudySampleType.IN_SAMPLE,
)

# 1. (Re)build the Tier-1 index over the archived top selection so we
#    can rank without decoding every bundle.
build_index(
    str(top_selection_path),
    show_progress=True,
    incremental=True,
)

# 2. Re-rank with the same BALANCED focus used in the in-sample sweep.
#    No limit — we want *all* promoted survivors for OOS evaluation.
top = rank_index(
    str(top_selection_path),
    focus=BacktestEvaluationFocus.BALANCED,
    engine="vector",
    study=in_sample_study,
    limit=None,
)

# 3. Materialise the winners (deduplicate by bundle_path so dual-engine
#    bundles aren't opened twice) and pull the *original* in-sample
#    grid variant out of ``metadata["params"]``. We use the metadata
#    payload — not ``bt.parameters`` — because notebook 03 hashed the
#    variant (minus ``_grid_profile``) to derive the in-sample
#    ``algorithm_id``. Hashing the same shape here is what makes the
#    OOS bundle collide with the in-sample envelope.
store = LocalDirStore(str(top_selection_path))
seen_bundles = set()
top_backtests = []
for row in top:
    bp = row["bundle_path"]
    if bp in seen_bundles:
        continue
    seen_bundles.add(bp)
    top_backtests.append(store.open(bp))

top_param_variations = []
skipped = []
for bt in top_backtests:
    md = bt.get_metadata() or {}
    variant = dict(md.get("params") or {})
    if not variant:
        # Legacy bundles without metadata["params"] fall back to the
        # canonical ``Backtest.parameters`` slot — the framework auto-
        # derived id will still match because both notebooks now use
        # the same hashing rule.
        variant = dict(bt.parameters or {})
    if not variant:
        skipped.append(bt.algorithm_id)
        continue
    top_param_variations.append(variant)

print(
    f"Loaded {len(top_param_variations)} param sets "
    f"from {len(top_backtests)} top-selection bundles"
)

if skipped:
    print(f"Skipped {len(skipped)} bundle(s) with no saved params: {skipped}")

## Strategy Initialization with Top Params

In [ ]:
from investing_algorithm_framework import generate_algorithm_id
from investing_algorithm_framework.domain import tqdm


def initialize_strategies(
    strategy_class,
    param_variations,
    symbols,
    market,
    trading_symbol="EUR",
    filter_fn=None,
):
    """Re-instantiate the in-sample winners for an out-of-sample run.

    Each ``variant`` is the in-sample grid variant we recovered from
    ``bt.metadata["params"]`` in the loader cell. We strip the
    underscore-prefixed metadata (``_grid_profile`` etc.) and hash the
    same stable subset notebook 03 used to derive the in-sample
    ``algorithm_id``. Passing that id explicitly here makes each OOS
    bundle land in the same ``<algorithm_id>.iafbt`` envelope as its
    in-sample winner (multi-study slot).
    """
    strategies = []

    for variant in tqdm(
        param_variations, desc="Initializing strategies", colour="green"
    ):
        # Mirror notebook 03: drop underscore-prefixed metadata before
        # hashing and before passing to the strategy constructor.
        strategy_params = {
            k: v for k, v in variant.items() if not k.startswith("_")
        }

        strategy = strategy_class(
            algorithm_id=generate_algorithm_id(params=strategy_params),
            symbols=symbols,
            trading_symbol=trading_symbol,
            market=market,
            metadata={
                "params": variant,
                "symbols": symbols,
                "market": market,
            },
            **strategy_params,
        )
        strategies.append(strategy)

    if filter_fn:
        strategies = [s for s in strategies if filter_fn(s)]

    return strategies


## Time Out-Sample Validation

In [ ]:
import os

from investing_algorithm_framework import create_app, RESOURCE_DIRECTORY, \
    DATA_DIRECTORY

strategies_time_oos = initialize_strategies(
    strategy_class=Strategy,
    param_variations=top_param_variations,
    symbols=time_oos_study.universe.symbols,
    market=time_oos_study.universe.market,
    trading_symbol=time_oos_study.universe.trading_symbol,
)

app = create_app(config={RESOURCE_DIRECTORY: "./resources", DATA_DIRECTORY: data_storage_path})

# ``study=time_oos_study`` carries its own ``Universe`` (BTC/ETH on
# BITVAVO/EUR), so the narrower Type-A descriptor is stamped on every
# bundle automatically — no separate ``universe=`` kwarg is needed.
# Notebook 05 can join in-sample vs. OOS results on the resulting
# ``universe_key`` even though the symbol set is a strict subset of
# the in-sample basket.
backtests_time_oos = app.run_backtest(
    strategies=strategies_time_oos,
    study=time_oos_study,
    continue_on_error=False,
    use_checkpoints=True,
    backtest_storage_directory=top_selection_path,
    show_progress=True,
    n_workers=os.cpu_count() - 3,
    dynamic_position_sizing=True,
)

## Universe out-sample validation

In [ ]:
import os

from investing_algorithm_framework import create_app, RESOURCE_DIRECTORY, DATA_DIRECTORY

strategies_universe_oos = initialize_strategies(
    strategy_class=Strategy,
    param_variations=top_param_variations,
    symbols=universe_oos_study.universe.symbols,
    market=universe_oos_study.universe.market,
    trading_symbol=universe_oos_study.universe.trading_symbol,
)

app = create_app(config={RESOURCE_DIRECTORY: "./resources", DATA_DIRECTORY: data_storage_path})

# ``study=universe_oos_study`` carries its own ``Universe`` (LINK/AVAX/
# ATOM/ALGO/XRP on BITVAVO/EUR), so the Type-B descriptor is stamped
# on every bundle automatically — no separate ``universe=`` kwarg is
# needed. Notebook 05 can join in-sample vs. OOS results on the
# resulting ``universe_key`` even though these symbols were never
# part of the in-sample basket.
backtests_universe_oos = app.run_backtest(
    strategies=strategies_universe_oos,
    study=universe_oos_study,
    continue_on_error=False,
    use_checkpoints=True,
    backtest_storage_directory=top_selection_path,
    show_progress=True,
    n_workers=os.cpu_count() - 3,
    dynamic_position_sizing=True,
)

In [ ]:
from pathlib import Path
from investing_algorithm_framework import (
    BacktestEvaluationFocus, build_index, rank_index, LocalDirStore,
)

backtest_results_dir = Path.cwd().parent / "backtest_results"
top_selection_path = backtest_results_dir / "top_selection"

# 1. Build (or refresh) the Tier-1 SQLite index for top selection
top_selection_index_path = build_index(
    str(top_selection_path),
    show_progress=True,
    incremental=True,
)

# 2. Re-rank the top selection with the same BALANCED focus used for
#    the in-sample sweep. ``study="in_sample_param_sweep"`` (a literal
#    name, not the ``in_sample_study`` object, so this cell doesn't
#    depend on an earlier cell having run) is required here: by now
#    ``top_selection_path`` also holds the time-OOS and universe-OOS
#    runs, and ``rank_index`` without a ``study=`` filter ranks across
#    every study side-by-side, which would score bundles on a mix of
#    in-sample and OOS metrics instead of the in-sample BALANCED score.
top = rank_index(
    str(top_selection_path),
    focus=BacktestEvaluationFocus.BALANCED,
    engine="vector",
    study="in_sample_param_sweep",
)

# 3. Materialise the ranked winners into ``Backtest`` objects so the
#    downstream metric-table helpers (which require ``Backtest``
#    instances, not raw SQLite index rows) can read per-run and

In [ ]:
from investing_algorithm_framework import (
    DEFAULT_TRADE_METRIC_COLUMNS,
    LocalDirStore,
    show_backtest_runs,
    show_backtest_summaries,
)

# Reload the bundles after both OOS studies have been persisted. This avoids
# inspecting stale Backtest objects materialised before the latest study run.
oos_store = LocalDirStore(str(top_selection_path))
oos_backtests = []
seen_bundle_paths = set()
for row in top:
    bundle_path = row["bundle_path"]
    if bundle_path in seen_bundle_paths:
        continue
    seen_bundle_paths.add(bundle_path)
    oos_backtests.append(oos_store.open(bundle_path))

for selected_study in (time_oos_study, universe_oos_study):
    show_backtest_summaries(
        oos_backtests,
        engine="vector",
        study=selected_study,
        sort_by="sharpe_ratio",
    )

    # Re-rank the same rows by cross-window robustness rather than
    # average return: a high Sharpe driven by one or two lucky windows
    # will fall here even though it topped the table above.
    show_backtest_summaries(
        oos_backtests,
        engine="vector",
        study=selected_study,
        sort_by="stability_score",
    )

    show_backtest_runs(
        oos_backtests,
        engine="vector",
        study=selected_study,
        columns=DEFAULT_TRADE_METRIC_COLUMNS,
        sort_by="profit_factor",
        page=1,
        page_size=25,
    )